# Первая ALS: обучение и warm-validation

Один стартовый опыт без перебора параметров: 64 фактора, 20 итераций,
regularization=0,1, alpha=20, seed=42, CPU (4 потока).

Матрица содержит `confidence = 1 + alpha × weight` для положительных взаимодействий.
Ненаблюдаемые пары имеют confidence=1 и целевой интерес 0. Веса берём из
`weights.py` без затухания: коэффициент оценки 0,2, planned/on_hold 0,5,
dropped и оценки ниже 5 не обучаются, но исключаются из выдачи.
Это стартовая конфигурация, не результат оптимизации.

Используем прежние границы и warm-validation. Изменение модели не меняет
пользователей и цели оценки. Последний test не оценивается.
[Документация implicit ALS](https://benfred.github.io/implicit/api/models/cpu/als.html).

In [1]:
import os
# Ограничиваем BLAS до импорта NumPy; параллелизм обучения задаётся отдельно.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate
from app.training.weights import weighted_interactions, DEFAULT_STATUS_WEIGHTS
from app.recsys.popularity import PopularityRecommender
from app.recsys.als import ALSRecommender

pd.set_option("display.float_format", "{:,.4f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
print({name: importlib.metadata.version(name) for name in ["implicit", "numpy", "scipy", "pandas"]})
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(history, min_user_items=5, min_item_users=5)
weighted = weighted_interactions(training)
counts = history.user_id.value_counts()
targets = {u: items for u, items in relevant_items(validation).items() if counts.get(u, 0) >= 5}
print(f"После фильтра: {len(training):,}; положительных взаимодействий: {len(weighted):,}")
print(f"Пользователей в warm-validation: {len(targets):,}")

/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'implicit': '0.7.3', 'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0'}


После фильтра: 3,499,908; положительных взаимодействий: 3,293,589
Пользователей в warm-validation: 7,184


## Обучение

Строки и столбцы получают плотные индексы независимо от внешних ID, включая
отрицательные ID внешних пользователей. Полная история передаётся отдельно.
После удаления нулевых весов у пользователя может не остаться взаимодействий:
чистая ALS вернёт ему пустой список, что считается нулевым результатом.

In [2]:
started = perf_counter()
model = ALSRecommender(factors=64, regularization=0.1, alpha=20,
                       iterations=20, random_state=42, num_threads=4).fit(weighted, history=history)
training_seconds = perf_counter() - started
print(f"Обучение: {training_seconds:.2f} с")
print("Факторы пользователей:", model.user_factors_.shape, "тайтлов:", model.item_factors_.shape)
unknown_users = sorted(set(targets) - set(model.user_index_))
print("Пользователей validation без факторов:", len(unknown_users))

Обучение: 7.36 с
Факторы пользователей: (8109, 64) тайтлов: (15799, 64)
Пользователей validation без факторов: 0


## Сравнение на одинаковых целях

Популярность за всё время и за 28 дней обучается на том же фильтрованном train,
без удаления низких оценок (это определение текущего baseline).
У всех моделей исключения — по полной истории до VALIDATION_START.

Вариант «ALS + fallback» только дополняет короткую/пустую выдачу свежей
популярностью, не смешивает её с полным топом ALS. Он показан отдельно,
чтобы не приписывать fallback качество самой модели.

In [3]:
started = perf_counter()
als_recs = model.recommend_many(targets, k=10)
inference_seconds = perf_counter() - started
recent = PopularityRecommender(window_days=28).fit(training, now=VALIDATION_START, history=history)
all_time = PopularityRecommender(window_days=None).fit(training, now=VALIDATION_START, history=history)
recent_recs = recent.recommend_many(targets, k=10)
all_time_recs = all_time.recommend_many(targets, k=10)
with_fallback = {}
for u, recs in als_recs.items():
    if len(recs) < 10:
        extra = [a for a in recent.recommend(u, k=10 + len(recs)) if a not in recs]
        with_fallback[u] = (recs + extra)[:10]
    else:
        with_fallback[u] = recs

seen = history.groupby("user_id").anime_id.agg(set).to_dict()
for recs_by_user in [als_recs, with_fallback]:
    assert all(len(recs) == len(set(recs)) and not (set(recs) & seen.get(u, set()))
               for u, recs in recs_by_user.items())
results = pd.DataFrame({
    "популярность за всё время": evaluate(all_time_recs, targets),
    "популярность 28 дней": evaluate(recent_recs, targets),
    "ALS": evaluate(als_recs, targets),
    "ALS + fallback 28 дней": evaluate(with_fallback, targets),
}).T
display(results)
print(f"Выдача ALS для {len(targets)} пользователей: {inference_seconds:.2f} с (офлайн, не HTTP p95)")
print("Пользователей с короткой выдачей ALS:", sum(len(r) < 10 for r in als_recs.values()))

,precision@10,recall@10,ndcg@10,coverage@10,users
популярность за всё время,0.0473,0.0281,0.0562,312.0000,"7,184.0000"
популярность 28 дней,0.0996,0.0645,0.1130,65.0000,"7,184.0000"
ALS,0.0818,0.0651,0.0998,"4,199.0000","7,184.0000"
ALS + fallback 28 дней,0.0818,0.0651,0.0998,"4,199.0000","7,184.0000"


Выдача ALS для 7184 пользователей: 9.12 с (офлайн, не HTTP p95)
Пользователей с короткой выдачей ALS: 0


## Сохранение и проверка артефакта

NPZ хранит факторы, параметры, исходные ID пользователей/тайтлов, полную маску
исключений среди известных модели тайтлов и контекст опыта. Загрузка без pickle.
Артефакт находится в `ml/artifacts/`, вне Git.

Это модель фиксированного снимка. При подключении к сайту новые взаимодействия
потребуют обновления исключений и периодического переобучения; здесь serving нет.

In [4]:
artifact = Path.cwd().parent / "artifacts" / "als_baseline.npz"
model.save(artifact, metadata={
    "snapshot_end": SNAPSHOT_END.isoformat(), "training_before": VALIDATION_START.isoformat(),
    "min_user_items": 5, "min_item_users": 5,
    "weights": {"status_weights": DEFAULT_STATUS_WEIGHTS, "score_coef": 0.2,
                "min_score": 5, "min_user_scores": 5, "clip": [0.5, 1.5], "half_life_days": None},
    "implicit_version": importlib.metadata.version("implicit"),
})
restored = ALSRecommender.load(artifact)
check_users = sorted(targets)[:20] + unknown_users[:5]
assert restored.recommend_many(check_users) == {u: als_recs[u] for u in check_users}
print("Артефакт:", artifact.name, f"({artifact.stat().st_size / 1024**2:.2f} MiB)")
print("Выдача после загрузки совпадает; отображения ID и исключения сохранены.")

Артефакт: als_baseline.npz (10.58 MiB)
Выдача после загрузки совпадает; отображения ID и исключения сохранены.


## Вывод первого запуска

- Обучение: 3 293 589 положительных взаимодействий, 8 109 пользователей,
  15 799 тайтлов. Для всех 7 184 пользователей warm-validation есть факторы
  и полный топ-10, поэтому fallback в этом опыте не понадобился.
- **ALS NDCG@10 ≈ 0,0998**, свежая популярность ≈ 0,1130,
  популярность за всё время ≈ 0,0562. Стартовая ALS пока уступает основному
  baseline по метрике выбора; победа ALS не предполагается заранее.
- Recall@10 у ALS ≈ 0,0651 против ≈ 0,0645 у свежей популярности,
  precision@10 ≈ 0,0818 против ≈ 0,0996.
- Покрытие ALS — 4 199 тайтлов против 65 у свежей популярности.
  Это разнообразие выдачи, а не самостоятельное доказательство её качества.
- Обучение заняло около 7 с, офлайн-выдача для всей группы — около 9 с
  в текущем окружении. Это не измерение HTTP p95 и не нагрузочный тест сервиса.
- Артефакт `als_baseline.npz` (~10,6 MiB) сохранён с ID и исключениями;
  проверена идентичность выдачи после загрузки. ID в этом опыте — **AniList**,
  а не локальные ID пользователей и тайтлов сайта.
- Следующий эксперимент: изолированно проверить коэффициент оценки
  0 / 0,2 / 0,4 при тех же остальных параметрах. При коэффициенте 0 низкие
  оценки по-прежнему исключаются — отключается относительная поправка.
  Далее — вес planned и параметры ALS, по одному изменению за раз.
- Test не оценивался. Параметры не выбираем по этому периоду.
